# Validation and Model Selection

Model selection belongs inside the development data. This notebook chooses polynomial degree by cross-validation and evaluates the selected model once on an untouched test set.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(8)
x = rng.uniform(-3, 3, 300)
y = 1 + 0.8*x - 0.5*x**2 + rng.normal(scale=1.2, size=x.size)
X = x[:, None]
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.25, random_state=17)
cv = KFold(n_splits=5, shuffle=True, random_state=19)
results = []
for degree in range(1, 7):
    model = make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), Ridge(alpha=1.0))
    mse = -cross_val_score(model, X_dev, y_dev, cv=cv, scoring='neg_mean_squared_error')
    results.append({'degree': degree, 'cv_mse': mse.mean()})
results = pd.DataFrame(results)
results


In [ ]:
best_degree = int(results.loc[results['cv_mse'].idxmin(), 'degree'])
final_model = make_pipeline(PolynomialFeatures(best_degree, include_bias=False), StandardScaler(), Ridge(alpha=1.0))
final_model.fit(X_dev, y_dev)
test_mse = mean_squared_error(y_test, final_model.predict(X_test))
best_degree, test_mse


The test set is not used to choose the degree. Keeping preprocessing inside the pipeline also prevents scaling information from leaking across validation folds.
